<h1>머신러닝</h1>

학습 방법: 지도학습, 비지도학습, 강화학습

문제 정의 -> 데이터 수집 -> 전처리 -> 학습 -> 평가 -> 배포

표준화: sandardization

정규화: Min-Max

<h2>전처리</h2>
결측치 확인(df.isna().sum())
결측치 채우기(fillna(), int.erpolate())

선형회귀

로지스틱 회귀(분류): 직선을 그어 두 무리를 가르고, 그 직선에서 얼마나 떠러졌는지를 0~1확률로 바꾼다

KNN: 결정트리: 직선이 아닌 때

KNN: 학습이 사실상 없다 : 데이터를 통째로 들고 있다가 물어볼 때 계산, 거리를 쓰므로 스케일 조정 필수

결정트리(예/아니오 질문을 이어감): 스케일 조정이 필요 없다, 싶이를 제한하지 않으면 반드시 과접합

비지도: 군집과 차원축소
1.   K_Means: K를 사람이 정해 줘야 한다, 중심을 다시 계산 -> 반복 -> 계산

요즘 유행: LightGBM / CatBoot (종류를 맞힌다, 성능우선)

In [1]:
#회귀
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression

# 1. 데이터 준비 (당뇨병 진행도 수치 데이터)
X, y = load_diabetes(return_X_y=True)

# 2. 모델 생성 및 학습 (X와 연속형 y 전달)
model = LinearRegression()
model.fit(X, y)

# 3. 예측 (새로운 X에 대한 연속적인 수치 출력)
predictions = model.predict(X[:5])

print("[회귀] 실제값:", y[:5])
print("[회귀] 예측값:", predictions)

[회귀] 실제값: [151.  75. 141. 206. 135.]
[회귀] 예측값: [206.11667725  68.07103297 176.88279035 166.91445843 128.46225834]


In [2]:
#분류
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier

# 1. 데이터 준비 (붓꽃 품종: 0, 1, 2)
X, y = load_iris(return_X_y=True)

# 2. 모델 생성 및 학습 (X와 범주형 y 전달)
model = DecisionTreeClassifier(max_depth=3)
model.fit(X, y)

# 3. 예측 (새로운 X가 속한 클래스 번호 출력)
predictions = model.predict(X[:5])

print("[분류] 실제 클래스:", y[:5])
print("[분류] 예측 클래스:", predictions)

[분류] 실제 클래스: [0 0 0 0 0]
[분류] 예측 클래스: [0 0 0 0 0]


물고기의 종류, 길이, 높이, 두께 등을 바탕으로 무게를 예측합니다. 변수 간의 상관관계가 놓아 다중공선성 문제를 학습하기 좋습니다.

회귀 코드만 사용

In [32]:
import pandas as pd
from sklearn.linear_model import LinearRegression

df = pd.read_csv('/content/sample_data/Fish.csv')
print(df.head())

df = pd.get_dummies(df, columns=['Species'], drop_first=True, dtype=int)

X = df.drop(columns='Weight')
y = df['Weight']


model = LinearRegression()
model.fit(X, y)


predictions = model.predict(X.head())

print("실제값:", y.head().values)
print("예측값:", predictions.round(1))

  Species  Weight  Length1  Length2  Length3   Height   Width
0   Bream   242.0     23.2     25.4     30.0  11.5200  4.0200
1   Bream   290.0     24.0     26.3     31.2  12.4800  4.3056
2   Bream   340.0     23.9     26.5     31.1  12.3778  4.6961
3   Bream   363.0     26.3     29.0     33.5  12.7300  4.4555
4   Bream   430.0     26.5     29.0     34.0  12.4440  5.1340
실제값: [242. 290. 340. 363. 430.]
예측값: [282.3 333.9 353.9 441.  439.3]


In [28]:
import pandas as pd

df = pd.read_csv('/content/sample_data/Fish.csv')
df.head()

,Species,Weight,Length1,Length2,Length3,Height,Width
0,Bream,242.0,23.2,25.4,30.0,11.5200,4.0200
1,Bream,290.0,24.0,26.3,31.2,12.4800,4.3056
2,Bream,340.0,23.9,26.5,31.1,12.3778,4.6961
3,Bream,363.0,26.3,29.0,33.5,12.7300,4.4555
4,Bream,430.0,26.5,29.0,34.0,12.4440,5.1340


In [39]:
from lightgbm import LGBMClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

iris = load_iris()

# 반환 순서: X_train, X_test, y_train, y_test
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

model = LGBMClassifier(verbose=-1)  # 학습 로그 경고 숨김
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print("Accuracy:", acc)

Accuracy: 1.0


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [48]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Colab 기본 제공 데이터 (train/test가 이미 나뉘어 있음)
train = pd.read_csv('/content/sample_data/california_housing_train.csv')
test  = pd.read_csv('/content/sample_data/california_housing_test.csv')

target = 'median_house_value'
X_train, y_train = train.drop(columns=target), train[target]
X_test,  y_test  = test.drop(columns=target),  test[target]

model = LGBMRegressor(random_state=42, verbose=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MSE :", round(mse, 2))
print("RMSE:", round(rmse, 2))
print("R2  :", round(r2, 4))

MSE : 2331923006.48
RMSE: 48289.99
R2  : 0.8177
